## calculate the chemeleon performance for the paper 

In [ ]:
from paper_dataset_config import DATASET_SETTINGS_UPDATE
import numpy as np
import pandas as pd
import math
from src.args import get_args
from src.peptide_fw import *
from benchmark_pepfw import *

output_dir = f'./results/paper/'
output_dir_split = f'{output_dir}/split/'

metrics_columns = ['series', 'num_peptides', 'split_type', 'fold', 'descriptor', 'model', 
                          'nrmse', 'r2', 'spearman_corr', 'pearson_corr', 'fine_tune_time', 
                          'train_time', 'test_time']
performance_metrics = pd.DataFrame(columns=metrics_columns)
target_column = 'potency'
num_splits = 5
for target_name in ['Target_A', 'Target_B', 'Target_C', 'Target_D', 'Target_E']:
    settings = DATASET_SETTINGS_UPDATE[target_name]
    series_sel_all = settings['series_sel']
    for series_sel in series_sel_all:
        print(f"Start evaluating {target_name}: {series_sel}")
        for split_type in ['cv', 'time', 'walkforward']:
            if split_type == 'walkforward':
                n_fold = num_splits-1
            else:
                n_fold = num_splits
            for db_fold_num in range(n_fold):
                pred_file = f"{output_dir}/chemeleon/{target_name}_{series_sel}/{split_type}_fold{db_fold_num}/predictions.csv"
                pred_df = pd.read_csv(pred_file)
                y_test = pred_df[target_column].values.flatten()
                y_pred = pred_df['predictions'].values.flatten()
                nrmse, r2, spearman_corr, pearson_corr = calc_metrics(y_test, y_pred.flatten())
                new_row = {
                    'series': f'{target_name}_{series_sel}',
                    'split_type': split_type,
                    'fold': db_fold_num,
                    'descriptor': 'graph', 
                    'model': 'chemeleon',
                    'nrmse': nrmse,
                    'r2': r2,
                    'spearman_corr': spearman_corr,
                    'pearson_corr': pearson_corr,
                    'fine_tune_time': 0,
                    'train_time': 'TBD', 
                    'test_time': 'TBD'
                }
                performance_metrics.loc[len(performance_metrics)] = new_row
performance_metrics.to_csv(f"{output_dir}/performance_metrics_chemeleon_all.csv", index=False)

Start evaluating Target_A: B-8154
Start evaluating Target_A: B-8154 Macrolactam
Start evaluating Target_A: B-8209
Start evaluating Target_B: 4R-5594
Start evaluating Target_B: 4R-2390
Start evaluating Target_B: 4R-9882
Start evaluating Target_C: L-300_Macrolactam
Start evaluating Target_C: L-320_Macrolactam
Start evaluating Target_C: L-878
Start evaluating Target_D: X-link 2-11
Start evaluating Target_E: B-3503
